In [1]:
# Mounting the drive -- input and output stored here
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
# Confirming file extraction
from pathlib import Path
import zipfile

ZIP_PATH = Path("/content/drive/MyDrive/SinoCT_files.zip")
DATA_DIR = Path("/content/SinoCT")
DATA_DIR.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(ZIP_PATH, "r") as zip_ref:
    zip_ref.extractall(DATA_DIR)

print("Extraction complete")
files = sorted(DATA_DIR.rglob("sinogram.npy"))
print("Sinogram files found:", len(files))
for file in files[:10]:
    print(file)

KeyboardInterrupt: 

KeyboardInterrupt: 

In [3]:

from pathlib import Path
import csv
import math
import time
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
from PIL import Image
from scipy.ndimage import binary_fill_holes, label
from skimage.filters import threshold_otsu
from skimage.metrics import structural_similarity
from skimage.transform import iradon

# ===================================================================
# 1. Input, output paths, setting and parameters
DATA_DIR = Path("/content/SinoCT")
GT_DIR = Path("/content/drive/MyDrive/gt_fullview_npy")
OUTPUT_DIR = Path("/content/drive/MyDrive/nerf")

TRAINING_SERIES = range(1009, 1030)
VALIDATION_SERIES = range(1030, 1040)
VIEW_COUNTS = [98]
SLICE_STEP = 4

# NeRF parameters
ITERATIONS = 3000
RAYS_PER_STEP = 512
POINTS_PER_RAY = 256
LEARNING_RATE = 0.0001
FOURIER_FREQUENCIES = 5
IMAGE_SIZE = 888
HELDOUT_VIEWS = 12
SEED = 42

# PNG settings
BG_LEVEL = 21
FALLBACK_PNG_SCALE = 0.01
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

COLUMNS = [
    "series", "slice", "method", "views", "runtime_s",
    "psnr", "ssim", "rmse", "psnr_head", "ssim_head",
    "rmse_head", "proj_psnr", "proj_ssim", "proj_rmse",
    "final_loss", "iters", "seed", "bg_level",
    "input_psnr", "input_ssim"
]

# ===================================================================
# 2. Load sinograms
def load_series(series_id):
    files = list(DATA_DIR.glob(
        f"**/series_{series_id}/sinogram/sinogram.npy"
    ))
    if len(files) != 1:
        raise FileNotFoundError(
            f"Expected one sinogram for series {series_id}; found {len(files)}"
        )
    data = np.load(files[0], mmap_mode="r")
    if data.ndim != 3 or data.shape[:2] != (984, 888):
        raise ValueError(f"Wrong sinogram shape: {data.shape}")
    return data

def view_selection(n_views):
    indices = np.round(
        np.linspace(0, 492, n_views, endpoint=False)
    ).astype(int)
    angles = np.linspace(0, 360, 984, endpoint=False)[indices]
    return indices, angles

# ===================================================================
# 3. NeRF model
class NeRFCT(nn.Module):
    def __init__(self):
        super().__init__()
        self.network = nn.Sequential(
            nn.Linear(2 + 4 * FOURIER_FREQUENCIES, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 128), nn.ReLU(),
            nn.Linear(128, 1)
        )
        nn.init.normal_(self.network[-1].weight, std=0.001)
        nn.init.constant_(self.network[-1].bias, -2.0)

    def forward(self, xy):
        features = [xy]
        for i in range(FOURIER_FREQUENCIES):
            frequency = (2 ** i) * torch.pi
            features += [
                torch.sin(frequency * xy),
                torch.cos(frequency * xy)
            ]
        encoded = torch.cat(features, dim=-1)
        return 0.01 * F.softplus(self.network(encoded))

# ===================================================================
# 4. project attenuation
def project_rays(model, angles, detectors):
    offset = (
        detectors.float() - (IMAGE_SIZE - 1) / 2
    ) * (2 / IMAGE_SIZE)
    half_chord = torch.sqrt(
        torch.clamp(1 - offset.square(), min=0)
    )
    intervals = torch.arange(
        POINTS_PER_RAY, device=DEVICE
    ).float()
    if model.training:
        t = (
            intervals[None] +
            torch.rand(len(detectors), POINTS_PER_RAY, device=DEVICE)
        ) / POINTS_PER_RAY
    else:
        t = (intervals[None] + 0.5) / POINTS_PER_RAY
    t = (2 * t - 1) * half_chord[:, None]
    c, s = torch.cos(angles), torch.sin(angles)
    x = offset[:, None] * c[:, None] + t * s[:, None]
    y = -offset[:, None] * s[:, None] + t * c[:, None]
    coordinates = torch.stack((x, y), dim=-1)
    attenuation = model(
        coordinates.reshape(-1, 2)
    ).reshape(-1, POINTS_PER_RAY)
    return attenuation.mean(dim=1) * half_chord * IMAGE_SIZE

# ===================================================================
# 5. Train by slice
def train_slice(full_sino, n_views):
    indices, degrees = view_selection(n_views)
    measured = torch.as_tensor(
        full_sino[:, indices].copy(), device=DEVICE
    )
    angles = torch.tensor(
        np.deg2rad(degrees.copy()),
        dtype=torch.float32, device=DEVICE
    )
    model = NeRFCT().to(DEVICE)
    optimiser = torch.optim.Adam(
        model.parameters(), lr=LEARNING_RATE
    )
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    start = time.perf_counter()
    model.train()
    for step in range(ITERATIONS):
        a = torch.randint(n_views, (RAYS_PER_STEP,), device=DEVICE)
        d = torch.randint(888, (RAYS_PER_STEP,), device=DEVICE)
        predicted = project_rays(model, angles[a], d)
        loss = F.mse_loss(predicted, measured[d, a])
        if not torch.isfinite(loss).item():
            raise RuntimeError("Non-finite training loss")
        optimiser.zero_grad()
        loss.backward()
        optimiser.step()
        if step == 0 or (step + 1) % 500 == 0:
            print(
                f"  Step {step+1}/{ITERATIONS}: "
                f"loss={loss.item():.6f}", flush=True
            )
    model.eval()
    with torch.no_grad():
        p = (
            (torch.arange(IMAGE_SIZE, device=DEVICE) + 0.5)
            * (2 / IMAGE_SIZE) - 1
        )
        yy, xx = torch.meshgrid(p, p, indexing="ij")
        coords = torch.stack((xx, yy), dim=-1).reshape(-1, 2)
        image = torch.cat([
            model(c).cpu().flatten()
            for c in coords.split(65536)
        ])
        image = image.reshape(IMAGE_SIZE, IMAGE_SIZE).numpy()
        image[(xx.square() + yy.square() > 1).cpu().numpy()] = 0
    if DEVICE.type == "cuda":
        torch.cuda.synchronize()
    runtime = time.perf_counter() - start
    return model, image.astype(np.float32), runtime, float(loss.item())

# ===================================================================
# 6. Image quality metrics
def evaluate_image(pred, gt):
    if pred.shape != gt.shape:
        raise ValueError("Prediction/ground-truth shape mismatch")

    span = float(gt.max() - gt.min())
    if span <= 0:
        raise ValueError("Ground truth has constant intensity")
    truth = (gt - gt.min()) / span
    estimate = (pred - gt.min()) / span
    error = (truth - estimate) ** 2
    mse = float(error.mean())

    ssim, ssim_map = structural_similarity(
        truth, estimate, data_range=1, full=True
    )
    mask = binary_fill_holes(
        truth > threshold_otsu(truth)
    )
    regions, count = label(mask)
    if count > 0:
        sizes = np.bincount(regions.ravel())
        sizes[0] = 0
        mask = regions == sizes.argmax()
    head_mse = (
        float(error[mask].mean())
        if mask.any() else float("nan")
    )
    return {
        "psnr": -10 * math.log10(mse) if mse > 0 else float("inf"),
        "ssim": float(ssim),
        "rmse": math.sqrt(mse),
        "psnr_head": -10 * math.log10(head_mse)
                     if head_mse > 0 else float("inf"),
        "ssim_head": float(ssim_map[mask].mean())
                     if mask.any() else float("nan"),
        "rmse_head": math.sqrt(head_mse)
    }

# ===================================================================
# 7. Evaluation at new angles
@torch.no_grad()
def evaluate_projections(model, full_sino, n_views):
    used, _ = view_selection(n_views)
    unused = np.setdiff1d(np.arange(492), used)
    heldout = unused[
        np.linspace(0, len(unused)-1, HELDOUT_VIEWS)
        .round().astype(int)
    ]
    all_angles = np.deg2rad(
        np.linspace(0, 360, 984, endpoint=False)
    )
    model.eval()
    columns = []
    for index in heldout:
        chunks = []
        for start in range(0, 888, 64):
            d = torch.arange(
                start, min(start+64, 888), device=DEVICE
            )
            angles = torch.full(
                (len(d),), float(all_angles[index]), device=DEVICE
            )
            chunks.append(
                project_rays(model, angles, d).cpu().numpy()
            )
        columns.append(np.concatenate(chunks))
    pred = np.stack(columns, axis=1)
    gt = full_sino[:, heldout]
    mse = float(np.mean((pred - gt)**2, dtype=np.float64))
    span = float(np.ptp(gt))
    return {
        "proj_psnr": 10 * math.log10(span**2 / mse)
                     if span > 0 and mse > 0 else float("nan"),
        "proj_ssim": float(structural_similarity(
            gt, pred, data_range=span
        )) if span > 0 else float("nan"),
        "proj_rmse": math.sqrt(mse)
    }

# ===================================================================
# 8. Reconstruction saving function
def png_scaling():
    levels = []
    for series in TRAINING_SERIES:
        path = GT_DIR / f"series_{series}_slice08.npy"
        if path.exists():
            gt = np.load(path, mmap_mode="r")
            if gt.shape == (IMAGE_SIZE, IMAGE_SIZE):
                levels.append(float(np.percentile(gt, 99.9)))
    if not levels:
        print("WARNING: Using fallback PNG scale 0.01")
    return max(
        max(levels) if levels else FALLBACK_PNG_SCALE,
        1e-8
    )
def save_pngs(image, name, display_scale):
    scaled = np.clip(image / display_scale, 0, 1)
    png = np.rint(
        BG_LEVEL + scaled * (255 - BG_LEVEL)
    ).astype(np.uint8)
    Image.fromarray(png, "L").save(
        OUTPUT_DIR / f"{name}.png"
    )
    foreground = image[image > 0]
    if foreground.size and foreground.max() > foreground.min():
        low, high = np.percentile(foreground, [1, 99])
        preview = np.clip(
            (image - low) / max(high - low, 1e-12), 0, 1
        )
        preview[image <= 0] = 0
    else:
        preview = np.zeros_like(image)
    Image.fromarray(
        np.rint(preview * 255).astype(np.uint8), "L"
    ).save(
        OUTPUT_DIR / "previews" / f"{name}_preview.png"
    )

# ===================================================================
# 9. Results and summary
def write_csv(rows):
    with open(OUTPUT_DIR / "nerf_results.csv", "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=COLUMNS)
        writer.writeheader()
        writer.writerows(rows)
    numerical = [
        x for x in COLUMNS
        if x not in ("series", "slice", "method", "views")
    ]
    with open(
        OUTPUT_DIR / "ner_results_summary.csv", "w", newline=""
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=["statistic"] + COLUMNS
        )
        writer.writeheader()
        for views in VIEW_COUNTS:
            group = [
                row for row in rows
                if int(row["views"]) == views
            ]
            if not group:
                continue
            for stat in ("mean", "std"):
                entry = {
                    "statistic": stat,
                    "method": "nerf",
                    "views": views
                }
                for col in numerical:
                    values = []
                    for row in group:
                        try:
                            value = float(row[col])
                            if np.isfinite(value):
                                values.append(value)
                        except (ValueError, TypeError, KeyError):
                            pass
                    if not values:
                        entry[col] = ""
                    elif stat == "mean":
                        entry[col] = float(np.mean(values))
                    elif len(values) > 1:
                        entry[col] = float(
                            np.std(values, ddof=1)
                        )
                    else:
                        entry[col] = ""
                writer.writerow(entry)

# ===================================================================
# 10. Validation
def main():
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    (OUTPUT_DIR / "previews").mkdir(exist_ok=True)
    print("Device:", DEVICE)
    print("Validation series:", list(VALIDATION_SERIES))
    print("Projection views:", VIEW_COUNTS)
    if not DATA_DIR.is_dir():
        raise FileNotFoundError(f"No input folder: {DATA_DIR}")
    if not GT_DIR.is_dir():
        print("WARNING: GT folder missing; CT metrics will be blank")
    scale = png_scaling()
    results_path = OUTPUT_DIR / "nerf_results.csv"
    if results_path.exists():
        with open(results_path, newline="") as f:
            reader = csv.DictReader(f)
            valid_headers = (
                set(COLUMNS),
                set(COLUMNS) | {"n_gaussians"}
            )
            if set(reader.fieldnames or []) not in valid_headers:
                raise ValueError("Unexpected existing CSV columns")
            rows = [
                {col: row.get(col, "") for col in COLUMNS}
                for row in reader
            ]
    else:
        rows = []
    if any(
        int(row["series"]) not in VALIDATION_SERIES
        for row in rows
    ):
        raise ValueError(
            "Existing CSV contains non-validation patients"
        )
    previous = {
        (int(r["series"]), int(r["slice"]), int(r["views"]))
        for r in rows
    }
    for series_id in VALIDATION_SERIES:
        series = load_series(series_id)
        print(f"\nSeries {series_id}: {series.shape[2]} slices")
        for slice_id in range(0, series.shape[2], SLICE_STEP):
            sino = np.asarray(
                series[:, :, slice_id].T, dtype=np.float32
            )
            if not np.isfinite(sino).all():
                raise ValueError("Sinogram contains NaN/Inf")
            gt_path = GT_DIR / (
                f"series_{series_id}_slice{slice_id:02d}.npy"
            )
            gt = (
                np.load(gt_path).astype(np.float32)
                if gt_path.exists() else None
            )
            for views in VIEW_COUNTS:
                name = (
                    f"series_{series_id}"
                    f"_slice{slice_id:02d}"
                    f"_v{views:03d}"
                )
                key = (series_id, slice_id, views)
                png = OUTPUT_DIR / f"{name}.png"
                preview = (
                    OUTPUT_DIR / "previews" /
                    f"{name}_preview.png"
                )
                if key in previous and png.exists() and preview.exists():
                    print("Skipping completed:", name)
                    continue
                print("Training", name, flush=True)
                model, image, seconds, loss = train_slice(
                    sino, views
                )
                if gt is not None:
                    image_scores = evaluate_image(image, gt)
                    idx, theta = view_selection(views)
                    fbp = iradon(
                        sino[:, idx],
                        theta=theta,
                        filter_name="ramp",
                        output_size=IMAGE_SIZE,
                        circle=True
                    ).astype(np.float32)
                    input_scores = evaluate_image(fbp, gt)
                else:
                    image_scores = {
                        m: float("nan") for m in (
                            "psnr", "ssim", "rmse",
                            "psnr_head", "ssim_head", "rmse_head"
                        )
                    }
                    input_scores = {
                        "psnr": float("nan"),
                        "ssim": float("nan")
                    }
                projection_scores = evaluate_projections(
                    model, sino, views
                )
                save_pngs(image, name, scale)
                result = {
                    "series": series_id,
                    "slice": slice_id,
                    "method": "nerf",
                    "views": views,
                    "runtime_s": round(seconds, 2),
                    **image_scores,
                    **projection_scores,
                    "final_loss": loss,
                    "iters": ITERATIONS,
                    "seed": SEED,
                    "bg_level": BG_LEVEL,
                    "input_psnr": input_scores["psnr"],
                    "input_ssim": input_scores["ssim"]
                }
                rows = [
                    r for r in rows
                    if (
                        int(r["series"]),
                        int(r["slice"]),
                        int(r["views"])
                    ) != key
                ]
                rows.append(result)
                write_csv(rows)
                previous.add(key)
                print(
                    f"  PSNR={result['psnr']:.2f}, "
                    f"SSIM={result['ssim']:.4f}, "
                    f"runtime={seconds:.1f}s",
                    flush=True
                )
                del model
                if DEVICE.type == "cuda":
                    torch.cuda.empty_cache()
    write_csv(rows)
    print("\nFull NeRF reconstruction complete!")
    print("Outputs saved in:", OUTPUT_DIR)

# ===================================================================
# 11. Model training
if __name__ == "__main__":
    main()


Device: cuda
Validation series: [1030, 1031, 1032, 1033, 1034, 1035, 1036, 1037, 1038, 1039]
Projection views: [98]

Series 1030: 32 slices
Skipping completed: series_1030_slice00_v098
Skipping completed: series_1030_slice04_v098
Skipping completed: series_1030_slice08_v098
Skipping completed: series_1030_slice12_v098
Skipping completed: series_1030_slice16_v098
Skipping completed: series_1030_slice20_v098
Skipping completed: series_1030_slice24_v098
Skipping completed: series_1030_slice28_v098

Series 1031: 40 slices
Skipping completed: series_1031_slice00_v098
Skipping completed: series_1031_slice04_v098
Skipping completed: series_1031_slice08_v098
Skipping completed: series_1031_slice12_v098
Skipping completed: series_1031_slice16_v098
Skipping completed: series_1031_slice20_v098
Skipping completed: series_1031_slice24_v098
Skipping completed: series_1031_slice28_v098
Skipping completed: series_1031_slice32_v098
Skipping completed: series_1031_slice36_v098

Series 1032: 40 slices
Sk

/tmp/ipykernel_1171/1107323647.py:302: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(png, "L").save(
/tmp/ipykernel_1171/1107323647.py:317: DeprecationWarning: 'mode' parameter is deprecated and will be removed in Pillow 13 (2026-10-15)
  Image.fromarray(


  PSNR=32.70, SSIM=0.9631, runtime=46.3s
Training series_1033_slice44_v098
  Step 1/3000: loss=1.033408
  Step 500/3000: loss=0.013991
  Step 1000/3000: loss=0.009020
  Step 1500/3000: loss=0.006599
  Step 2000/3000: loss=0.005998
  Step 2500/3000: loss=0.006849
  Step 3000/3000: loss=0.009539
  PSNR=34.70, SSIM=0.9736, runtime=48.3s

Series 1034: 40 slices
Training series_1034_slice00_v098
  Step 1/3000: loss=1.629563
  Step 500/3000: loss=0.051711
  Step 1000/3000: loss=0.024668
  Step 1500/3000: loss=0.023232
  Step 2000/3000: loss=0.013501
  Step 2500/3000: loss=0.012983
  Step 3000/3000: loss=0.013658
  PSNR=31.28, SSIM=0.9429, runtime=50.3s
Training series_1034_slice04_v098
  Step 1/3000: loss=2.150164
  Step 500/3000: loss=0.096345
  Step 1000/3000: loss=0.037413
  Step 1500/3000: loss=0.020704
  Step 2000/3000: loss=0.019475
  Step 2500/3000: loss=0.021740
  Step 3000/3000: loss=0.016885
  PSNR=31.00, SSIM=0.9390, runtime=49.4s
Training series_1034_slice08_v098
  Step 1/3000: l